# MDC Preprocessing (MERGED, KAGGLE) -- single pipeline, binary + multiclass + timestamps

**This notebook replaces both `mdc_preprocess_vNext_kaggle.ipynb` and the old (non-merged) version of this file.** One run produces everything downstream needs:

- `windows_vnext.npz` **and** `windows_vnext_mc.npz` (byte-identical copies -- kept under both names so `mdc_drift_aware_kaggle` and `mdc_model_vNext_kaggle`'s per-attack-type eval both find what they already look for)
- `drift_baseline_vnext.npz` (self-sufficient -- no second notebook needed)
- `preproc_vnext.pkl`, `manifest_vnext.json`, `mdc_label_map.json`

**Kaggle-native:** no Google Drive, no Colab. See the handoff pattern below (attach a previous notebook's output via '+ Add Data', or upload its downloaded zip as a Kaggle Dataset).

**Old file retired:** `mdc_preprocess_vNext_kaggle.ipynb` is superseded -- do not run it, kept only for reference.

**Pipeline run order:** this notebook -> `mdc_model_vNext_kaggle` -> `mdc_drift_aware_kaggle` -> `mdc_baselines_kaggle`.


## 0. Setup & config

In [ ]:
import sys, os, subprocess, gc, json
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
import warnings
warnings.filterwarnings('ignore')

_IN_KAGGLE = os.path.exists('/kaggle/working')

if _IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'kagglehub[pandas-datasets]', 'joblib'], check=False)
    OUTPUT_DIR = '/kaggle/working/processed'
else:
    OUTPUT_DIR = os.path.normpath('../outputs/vNEXT_test/processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

VERSION       = 'vnext'
RANDOM_STATE  = 42
BENIGN_LABEL  = 0

TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15
TEST_RATIO    = 0.15

MIN_FLOWS          = 10_000
GAP_THRESHOLD      = 60
VARIANCE_THRESHOLD = 0.01
CORR_THRESHOLD     = 0.95

BUCKET_FREQ           = '15s'
BUCKET_AGG            = 'mean_max_std'
WINDOW_SIZE           = 10
STRIDE                = 2
ATTACK_FRAC_THRESHOLD = 0.5
MAX_GAP_BUCKETS       = 4
POST_SCALE_CLIP       = 10.0

DROP_COLS = [
    'Flow ID', 'Dst IP', 'Timestamp',
    'Fwd URG Flags', 'Bwd URG Flags',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count',
]

print(f'preprocess_{VERSION}_mc config')
print(f'  IN_KAGGLE        = {_IN_KAGGLE}')
print(f'  OUTPUT_DIR       = {OUTPUT_DIR}')
print(f'  bucket_agg       = {BUCKET_AGG}')
print(f'  window           = {WINDOW_SIZE} x {BUCKET_FREQ} = {WINDOW_SIZE*15}s')
print(f'  stride           = {STRIDE} x {BUCKET_FREQ} = {STRIDE*15}s')
print(f'  attack_frac_thr  = {ATTACK_FRAC_THRESHOLD}')
print(f'  max_gap_buckets  = {MAX_GAP_BUCKETS}')
print(f'  post_scale_clip  = +/-{POST_SCALE_CLIP}')
print(f'  MC labels        : y_val_multiclass + y_test_multiclass + ts_*')


## 1. Load raw data (Kaggle or local CSV)

In [ ]:
from pathlib import Path
import glob

KAGGLE_DATASET = 'yigitsever/misuse-detection-in-containers-dataset'
KAGGLE_CSV     = 'MDC dataset.csv'

# If you attached this dataset via '+ Add Data' in the Kaggle UI, it's already
# under /kaggle/input -- check there first (fast, no download/auth needed).
_input_hits = glob.glob('/kaggle/input/**/*.csv', recursive=True) if _IN_KAGGLE else []
_input_hits = [h for h in _input_hits
               if 'mdc' in h.lower() or 'misuse' in h.lower() or 'dataset' in h.lower()]

if _input_hits:
    _pick = sorted(_input_hits, key=lambda p: os.path.getsize(p), reverse=True)[0]
    df = pd.read_csv(_pick)
    print(f'Loaded from attached input data: {_pick}')
else:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    try:
        df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, KAGGLE_DATASET, KAGGLE_CSV)
    except Exception:
        root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        candidates = sorted(root.rglob('*.csv'), key=lambda p: p.stat().st_size, reverse=True)
        df = pd.read_csv(candidates[0])
    print('Loaded via kagglehub download (needs internet enabled on this kernel).')

df.columns = df.columns.str.strip()
_labels = sorted(df['Label'].unique())
print(f'Raw shape: {df.shape}   labels={_labels}')


## 2. Container filter + timestamp + session assignment

In [ ]:
counts   = df['Src IP'].value_counts()
keep_ips = counts[counts >= MIN_FLOWS].index.tolist()
df       = df[df['Src IP'].isin(keep_ips)].copy().reset_index(drop=True)
print(f'After container filter: {len(df):,} rows  /  {len(keep_ips)} containers')

df['ts'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df = df.dropna(subset=['ts']).sort_values(['Src IP', 'ts']).reset_index(drop=True)

def assign_sessions(g, threshold=60):
    gap = g['ts'].diff().dt.total_seconds().fillna(0)
    sn  = (gap > threshold).cumsum()
    g['session_id'] = g['Src IP'].astype(str) + '_s' + sn.astype(str)
    return g

df = df.groupby('Src IP', group_keys=False).apply(assign_sessions, threshold=GAP_THRESHOLD)
df['time_gap_s'] = df.groupby('Src IP')['ts'].diff().dt.total_seconds()
_n_sess = df['session_id'].nunique()
print(f'Sessions: {_n_sess:,}')
print(df.groupby('Src IP')['session_id'].nunique().to_string())

## 3. Session-level random split + benign-only fit setup

Random per-container split (TRAIN_RATIO=0.70) → train ∪ holdout. All subsequent statistics
(median, variance, correlation, clip, scaler) are fit on **training benign flows only**.

In [ ]:
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
gc.collect()

_ss = df[['Src IP', 'session_id', 'ts']].drop_duplicates('session_id').copy()
rng = np.random.default_rng(RANDOM_STATE)
train_sessions, holdout_sessions = set(), set()
for src_ip, grp in _ss.groupby('Src IP'):
    sess = grp['session_id'].tolist()
    n_train = max(1, int(len(sess) * TRAIN_RATIO))
    shuffled = rng.permutation(sess)
    train_sessions.update(shuffled[:n_train])
    holdout_sessions.update(shuffled[n_train:])
del _ss; gc.collect()

train_mask  = df['session_id'].isin(train_sessions)
meta_cols   = ['Src IP', 'session_id', 'Label', 'ts']
meta_train  = df.loc[train_mask, meta_cols].reset_index(drop=True)
meta_holdout= df.loc[~train_mask, meta_cols].reset_index(drop=True)

non_meta = [c for c in df.columns if c not in meta_cols + ['time_gap_s']]
df_train = df.loc[train_mask, non_meta].copy().reset_index(drop=True)
df_hold  = df.loc[~train_mask, non_meta].copy().reset_index(drop=True)
del df; gc.collect()

benign_idx = meta_train['Label'].eq(BENIGN_LABEL).values
_hold_atk_pct = (meta_holdout['Label'] != 0).mean()*100
print(f'train flows: {len(df_train):,}   benign: {benign_idx.sum():,}  ({benign_idx.mean()*100:.1f}%)')
print(f'hold  flows: {len(df_hold):,}    holdout attack rate: {_hold_atk_pct:.1f}%')

## 4. Inf/NaN/dtype hygiene

In [ ]:
def to_numeric_safe(df_):
    for c in df_.columns:
        df_[c] = pd.to_numeric(df_[c], errors='coerce').astype(np.float32)
    return df_

df_train = to_numeric_safe(df_train)
df_hold  = to_numeric_safe(df_hold)

df_train = df_train.replace([np.inf, -np.inf], np.nan)
df_hold  = df_hold .replace([np.inf, -np.inf], np.nan)

train_medians = df_train[benign_idx].median(numeric_only=True).fillna(0.0).astype(np.float32)
df_train = df_train.fillna(train_medians)
df_hold  = df_hold .fillna(train_medians)
print(f'Filled NaNs with benign-train medians ({len(train_medians)} columns)')

## 5. Variance threshold filter (fit on benign train)

In [ ]:
feat_names_before_var = df_train.columns.tolist()
vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
vt.fit(df_train.loc[benign_idx].to_numpy(dtype=np.float32))
var_mask = vt.get_support()
keep_var = [c for c, ok in zip(feat_names_before_var, var_mask) if ok]
df_train = df_train[keep_var]; df_hold = df_hold[keep_var]
print(f'VarianceThreshold {VARIANCE_THRESHOLD} -> kept {df_train.shape[1]} / {len(feat_names_before_var)}')

## 6. Correlation filter (benign-train sample)

In [ ]:
n_sample = min(100_000, int(benign_idx.sum()))
sample   = df_train.loc[benign_idx].sample(n=n_sample, random_state=RANDOM_STATE).astype(np.float32)
corr     = sample.corr(numeric_only=True).abs()
upper    = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr= [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
df_train = df_train.drop(columns=drop_corr, errors='ignore')
df_hold  = df_hold .drop(columns=drop_corr, errors='ignore')
del sample, corr, upper; gc.collect()
print(f'Correlation>{CORR_THRESHOLD}: dropped {len(drop_corr)}  ->  {df_train.shape[1]} cols')

## 7. Clip bounds (IQR-3 / p99) — fit on benign train

In [ ]:
df_tr_b = df_train.loc[benign_idx]
clip_bounds, n_iqr, n_p99 = {}, 0, 0
for col in df_train.columns:
    s = df_tr_b[col]
    if s.nunique(dropna=False) <= 1:
        clip_bounds[col] = (None, float(abs(s.iloc[0])) if len(s) else None, 'const_cap')
        n_p99 += 1; continue
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    if iqr > 0:
        clip_bounds[col] = (q1 - 3*iqr, q3 + 3*iqr, 'iqr'); n_iqr += 1
    else:
        p99 = float(s.quantile(0.99))
        clip_bounds[col] = (None, p99 if p99 > 0 else None, 'p99'); n_p99 += 1
for col, (lo, hi, st) in clip_bounds.items():
    if st == 'const_cap' and hi is not None:
        df_train[col] = df_train[col].clip(upper=hi); df_hold[col] = df_hold[col].clip(upper=hi)
    elif st in ('iqr', 'p99'):
        df_train[col] = df_train[col].clip(lower=lo, upper=hi)
        df_hold [col] = df_hold [col].clip(lower=lo, upper=hi)
print(f'IQR-clipped {n_iqr}, p99/const-clipped {n_p99}')

## 8. Flow-level StandardScaler (benign train) + clip ±10
RobustScaler was discarded — on benign-only flows the IQR can be ≈ 0 → values ~1e5 (v3 run 2 evidence).

In [ ]:
feat_cols_final = df_train.columns.tolist()
X_tr = df_train.to_numpy(dtype=np.float32, copy=True)
X_ho = df_hold .to_numpy(dtype=np.float32, copy=True)
del df_train, df_hold; gc.collect()

scaler_flow = StandardScaler().fit(X_tr[benign_idx])
X_tr = np.clip(scaler_flow.transform(X_tr), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
X_ho = np.clip(scaler_flow.transform(X_ho), -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
_sc_max = max(np.abs(X_tr).max(), np.abs(X_ho).max())
print(f'flow scaler fit on {int(benign_idx.sum())} benign flows  -- max|X|={_sc_max:.3f}')

df_tr_proc = pd.DataFrame(X_tr, columns=feat_cols_final, index=meta_train.index)
df_tr_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_train[['Src IP','session_id','Label','ts']].values
df_ho_proc = pd.DataFrame(X_ho, columns=feat_cols_final, index=meta_holdout.index)
df_ho_proc[['Src IP', 'session_id', 'Label', 'ts']] = meta_holdout[['Src IP','session_id','Label','ts']].values
del X_tr, X_ho; gc.collect()

## 9. 15-second bucketing — mean + max + std + flow_count + label_multiclass

In [ ]:
def bucket_flows(df_proc, feature_cols, bucket_freq, benign_label, flow_count_max=None):
    df_proc = df_proc.copy()
    df_proc['ts']     = pd.to_datetime(df_proc['ts'])
    df_proc['bucket'] = df_proc['ts'].dt.floor(bucket_freq)
    g = ['Src IP', 'session_id', 'bucket']
    feat_agg = df_proc.groupby(g, observed=True)[feature_cols].agg(['mean','max','std'])
    feat_agg.columns = [f'{c}_{s}' for c, s in feat_agg.columns]
    feat_agg = feat_agg.reset_index()
    std_cols = [c for c in feat_agg.columns if c.endswith('_std')]
    feat_agg[std_cols] = feat_agg[std_cols].fillna(0.0)
    cnt = df_proc.groupby(g, observed=True).size().reset_index(name='_flow_count_raw')
    # binary label: 0=benign, 1=attack
    lbl = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: 0 if (x == benign_label).all() else 1).reset_index(name='label')
    # dominant original label per bucket (0-11)
    lbl_mc = df_proc.groupby(g, observed=True)['Label'].apply(
        lambda x: int(x.mode().iloc[0])).reset_index(name='label_multiclass')
    out = feat_agg.merge(cnt, on=g).merge(lbl, on=g).merge(lbl_mc, on=g)
    if flow_count_max is None:
        flow_count_max = float(out['_flow_count_raw'].quantile(0.99))
    out['flow_count'] = (out['_flow_count_raw'] / flow_count_max).clip(upper=1.0).astype(np.float32)
    out = out.drop(columns=['_flow_count_raw'])
    # exclude both label columns from feature list
    bucket_feature_cols = [c for c in out.columns if c not in g + ['label', 'label_multiclass']]
    return out, flow_count_max, bucket_feature_cols

train_b, flow_count_max, bucket_feature_cols = bucket_flows(
    df_tr_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=None)
hold_b, _, _ = bucket_flows(
    df_ho_proc, feat_cols_final, BUCKET_FREQ, BENIGN_LABEL, flow_count_max=flow_count_max)
del df_tr_proc, df_ho_proc; gc.collect()

_tb_atk = int(train_b['label'].sum())
_tb_pct = train_b['label'].mean()*100
_hb_atk = int(hold_b['label'].sum())
_hb_pct = hold_b['label'].mean()*100
print(f'Train   buckets: {len(train_b):,}   attack: {_tb_atk:,} ({_tb_pct:.1f}%)')
print(f'Holdout buckets: {len(hold_b):,}    attack: {_hb_atk:,} ({_hb_pct:.1f}%)')
print(f'Features/bucket: {len(bucket_feature_cols)}  ({len(feat_cols_final)} flow cols x 3 + flow_count)')
_mc_uniq = sorted(train_b['label_multiclass'].unique().tolist())
print(f'label_multiclass unique labels (train): {_mc_uniq}')

## 10. Short-gap fill + bucket-level p1/p99 clip + bucket scaler

In [ ]:
bucket_fill = (train_b.loc[train_b['label']==0, bucket_feature_cols]
               .median().fillna(0.0).astype(np.float32))

def short_gap_fill(bdf, freq, feat_cols, fill_vals, max_gap=MAX_GAP_BUCKETS):
    f = pd.Timedelta(freq); span = max_gap * f
    def per_session(g):
        g = g.sort_values('bucket').drop_duplicates(subset=['bucket'])
        src, sid = g['Src IP'].iloc[0], g['session_id'].iloc[0]
        rows, prev = [], None
        for _, r in g.iterrows():
            if prev is not None:
                gap = r['bucket'] - prev
                if f < gap <= span + f:
                    for tb in pd.date_range(prev + f, r['bucket'] - f, freq=freq):
                        rec = {c: float(fill_vals[c]) for c in feat_cols}
                        rec.update({'Src IP': src, 'session_id': sid, 'bucket': tb,
                                    'label': 0, 'label_multiclass': 0})
                        rows.append(rec)
            rows.append(r.to_dict()); prev = r['bucket']
        return pd.DataFrame(rows)
    return (bdf.groupby(['Src IP','session_id'], group_keys=False)
               .apply(per_session).reset_index(drop=True))

train_b = short_gap_fill(train_b, BUCKET_FREQ, bucket_feature_cols, bucket_fill)
hold_b  = short_gap_fill(hold_b , BUCKET_FREQ, bucket_feature_cols, bucket_fill)
print(f'After short-gap fill (<= {MAX_GAP_BUCKETS} buckets) -- '
      f'train: {len(train_b):,}  hold: {len(hold_b):,}')

# Bucket-level clip from benign train p1/p99
_b = train_b.loc[train_b['label']==0, bucket_feature_cols]
lo = _b.quantile(0.01).astype(np.float32); hi = _b.quantile(0.99).astype(np.float32)
for c in bucket_feature_cols:
    L, H = float(lo[c]), float(hi[c])
    if not (np.isfinite(L) and np.isfinite(H) and L < H): continue
    train_b[c] = train_b[c].clip(L, H); hold_b[c] = hold_b[c].clip(L, H)
del _b; gc.collect()

# Bucket-level StandardScaler on benign train buckets
scaler_bucket = StandardScaler().fit(
    train_b.loc[train_b['label']==0, bucket_feature_cols].to_numpy(dtype=np.float32))
for bdf in (train_b, hold_b):
    arr = scaler_bucket.transform(bdf[bucket_feature_cols].to_numpy(dtype=np.float32))
    bdf[bucket_feature_cols] = np.clip(arr, -POST_SCALE_CLIP, POST_SCALE_CLIP).astype(np.float32)
_bsc_max = float(np.abs(train_b[bucket_feature_cols].values).max())
print(f'Bucket scaler fit on benign train buckets  -- max|X|={_bsc_max:.3f}')

## 11. Sliding windows (T=10, stride=2) — multiclass + window_end_ts

Returns binary label, container ID, dominant multiclass label, and **unix end-timestamp**
of each window (last bucket in the window). Timestamps enable `ordering_mode=timestamp`
in the drift-aware streaming replay.


In [ ]:
def make_windows(bucket_df, W, S, feat_cols, thr=ATTACK_FRAC_THRESHOLD):
    """Sliding windows with multiclass labels + per-window end timestamp."""
    Xs, Ys, Cs, Ys_mc, Ts = [], [], [], [], []
    for (src, sid), grp in bucket_df.groupby(['Src IP', 'session_id'], observed=True):
        grp = grp.sort_values('bucket')
        arr = grp[feat_cols].to_numpy(dtype=np.float32)
        lbl = grp['label'].to_numpy()
        lbl_mc = grp['label_multiclass'].to_numpy()
        buckets = pd.to_datetime(grp['bucket']).to_numpy()
        for k in range(0, len(arr) - W + 1, S):
            seg = lbl[k:k + W]
            seg_mc = lbl_mc[k:k + W]
            Xs.append(arr[k:k + W])
            Ys.append(int((seg != 0).mean() >= thr))
            Cs.append(src)
            Ts.append(buckets[k + W - 1])  # window_end_ts
            if (seg == 0).all():
                Ys_mc.append(0)
            else:
                atk = seg_mc[seg_mc != 0]
                vals, cnts = np.unique(atk, return_counts=True)
                Ys_mc.append(int(vals[cnts.argmax()]))
    if not Xs:
        empty_x = np.empty((0, W, len(feat_cols)), np.float32)
        empty_i8 = np.empty(0, np.int8)
        empty_obj = np.empty(0, object)
        empty_ts = np.empty(0, np.int64)
        return empty_x, empty_i8, empty_obj, empty_i8, empty_ts
    ts_unix = (pd.to_datetime(np.asarray(Ts)).astype('int64') // 10**9).astype(np.int64)
    return (np.asarray(Xs, np.float32), np.asarray(Ys, np.int8),
            np.asarray(Cs, object), np.asarray(Ys_mc, np.int8), ts_unix)

X_train_all, y_train_all, c_train_all, ymc_train_all, ts_train_all = make_windows(
    train_b, WINDOW_SIZE, STRIDE, bucket_feature_cols)
X_hold, y_hold, c_hold, ymc_hold, ts_hold = make_windows(
    hold_b, WINDOW_SIZE, STRIDE, bucket_feature_cols)
del train_b, hold_b; gc.collect()

benign_w = y_train_all == BENIGN_LABEL
X_train = X_train_all[benign_w]
y_train = y_train_all[benign_w]
c_train = c_train_all[benign_w]
del X_train_all, y_train_all, c_train_all, ymc_train_all, ts_train_all; gc.collect()

print(f'Train benign windows : {X_train.shape}')
print(f'Holdout windows      : {X_hold.shape}  attack rate {y_hold.mean()*100:.1f}%')
_hold_mc_uniq = sorted(np.unique(ymc_hold).tolist())
print(f'Holdout multiclass labels: {_hold_mc_uniq}')
print(f'Holdout ts unix range    : [{int(ts_hold.min())}, {int(ts_hold.max())}]  '
      f'span={(ts_hold.max()-ts_hold.min())/3600:.1f}h')


## 12. Stratified val / test split by (container × label)

Keeps `ts_val` / `ts_test` aligned with the same stratified split used for features and labels.


In [ ]:
from sklearn.model_selection import train_test_split
strat_key = pd.Series(c_hold.astype(str) + '|' + y_hold.astype(str))
vc = strat_key.value_counts()
_safe_key = strat_key.where(strat_key.map(vc).ge(2), other='_other')
X_val, X_test, y_val, y_test, c_val, c_test, ymc_val, ymc_test, ts_val, ts_test = train_test_split(
    X_hold, y_hold, c_hold, ymc_hold, ts_hold,
    test_size=0.5, stratify=_safe_key, random_state=RANDOM_STATE)
del X_hold, y_hold, c_hold, ymc_hold, ts_hold; gc.collect()

print(f'X_val  : {X_val.shape}   attack {y_val.mean()*100:.1f}%   containers={len(set(c_val.tolist()))}')
print(f'X_test : {X_test.shape}  attack {y_test.mean()*100:.1f}%   containers={len(set(c_test.tolist()))}')
_vmc_uniq = sorted(np.unique(ymc_val).tolist())
_tmc_uniq = sorted(np.unique(ymc_test).tolist())
print(f'y_val  multiclass labels: {_vmc_uniq}')
print(f'y_test multiclass labels: {_tmc_uniq}')
print(f'ts_test: unix [{int(ts_test.min())}, {int(ts_test.max())}]  '
      f'span={(ts_test.max()-ts_test.min())/3600:.1f}h')


## 13. Pre-save validation gates

In [ ]:
_xmax  = float(max(np.abs(X_train).max(), np.abs(X_val).max(), np.abs(X_test).max()))
_xmean = float(np.abs(X_train).mean())
_nan   = bool(np.isnan(X_train).any() or np.isnan(X_val).any() or np.isnan(X_test).any())

print('Window scale check')
print(f'  mean|X_train|={_xmean:.4f}  max|X|={_xmax:.4f}  nan={_nan}')
if _nan or _xmax > 50 or _xmean > 20:
    raise ValueError(f'BAD SCALE: re-run from \u00a78 StandardScaler + clip; max|X|={_xmax}')
if (y_train != 0).any():
    raise ValueError('Train windows must be benign-only')

overlap = train_sessions & holdout_sessions
assert len(overlap) == 0, 'Session leakage between train and holdout!'
print('Session leakage      : PASS')
print(f'Train benign-only    : PASS ({(y_train==0).all()})')
print(f'Holdout attack rate  : {y_val.mean()*100:.1f}% val  /  {y_test.mean()*100:.1f}% test')

# MC sanity: every attack window must have a non-zero multiclass label
_mc_bad = int(((y_test == 1) & (ymc_test == 0)).sum())
assert _mc_bad == 0, f'{_mc_bad} attack windows have label_multiclass=0'
print(f'MC label sanity      : PASS (no attack window with multiclass=0)')

# Timestamp integrity (Task 6) — required for temporal streaming replay
assert len(ts_val) == len(y_val) and len(ts_test) == len(y_test), 'ts length mismatch'
assert np.isfinite(ts_test.astype(np.float64)).all(), 'ts_test has non-finite values'
assert np.nanstd(ts_test.astype(np.float64)) > 0, 'ts_test has zero variance'
print(f'Timestamp integrity  : PASS  ts_test span={(ts_test.max()-ts_test.min())/3600:.1f}h')


## 14. Drift baseline (unchanged from vNext — for reference)

Per-feature mean / std / quantiles of benign train — used by model to compute PSI and KS.

In [ ]:
F = X_train.shape[2]
flat_train = X_train.reshape(-1, F)
drift_quantiles = np.linspace(0.0, 1.0, 11)
drift_baseline = {
    'mean'      : flat_train.mean(axis=0).astype(np.float32),
    'std'       : flat_train.std(axis=0).astype(np.float32) + 1e-6,
    'q'         : np.quantile(flat_train, drift_quantiles, axis=0).astype(np.float32),
    'q_levels'  : drift_quantiles.astype(np.float32),
    'feat_names': bucket_feature_cols,
}
print(f'drift baseline: features={F}  quantile levels={len(drift_quantiles)}')
print(f'  mean range  [{drift_baseline["mean"].min():.3f}, {drift_baseline["mean"].max():.3f}]')
print(f'  std  range  [{drift_baseline["std"].min():.3f},  {drift_baseline["std"].max():.3f}]')
print('NOTE: drift baseline is identical to vNext run — only saved here for completeness.')

## 15. Save all artifacts (single merged pipeline)

Produces every file every downstream notebook needs, in one run -- see the title cell for the full list.


In [ ]:
import joblib
import shutil

# Single merged npz -- contains everything: binary labels + multiclass labels +
# timestamps. Saved under BOTH filenames so every existing downstream notebook's
# search succeeds with zero changes: mdc_drift_aware looks for 'windows_vnext.npz'
# specifically; mdc_model_vNext's §19 per-attack-type eval looks for
# 'windows_vnext_mc.npz' specifically. The second file is a byte-identical copy,
# not a recompute, so there is no risk of the two ever diverging.
npz_path = f'{OUTPUT_DIR}/windows_{VERSION}.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_val=y_val,
    y_test=y_test,
    y_val_multiclass=ymc_val.astype(np.int8),
    y_test_multiclass=ymc_test.astype(np.int8),
    c_val=c_val.astype(str),
    c_test=c_test.astype(str),
    ts_val=ts_val.astype(np.int64),
    ts_test=ts_test.astype(np.int64),
)
_sz = os.path.getsize(npz_path) / 1024 / 1024
print(f'Saved windows_{VERSION}.npz -> {npz_path}  ({_sz:.1f} MB)')
print('Keys: X_train, X_val, X_test, y_val, y_test,')
print('      y_val_multiclass, y_test_multiclass, c_val, c_test, ts_val, ts_test')
print(f'ts_test unix range: [{int(ts_test.min())}, {int(ts_test.max())}]')

npz_mc_path = f'{OUTPUT_DIR}/windows_{VERSION}_mc.npz'
shutil.copy2(npz_path, npz_mc_path)
print(f"Copied -> {npz_mc_path}  (identical content, kept for filename "
      f"compatibility with the model notebook's per-attack-type eval)")

drift_path = f'{OUTPUT_DIR}/drift_baseline_{VERSION}.npz'
np.savez_compressed(drift_path, **drift_baseline)
print(f'Saved drift    -> {drift_path}')

pkl_path = f'{OUTPUT_DIR}/preproc_{VERSION}.pkl'
joblib.dump({
    'version': VERSION, 'created_at': datetime.now(timezone.utc).isoformat(),
    'random_state': RANDOM_STATE,
    'window_size': WINDOW_SIZE, 'stride': STRIDE, 'bucket_freq': BUCKET_FREQ,
    'bucket_agg': BUCKET_AGG, 'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
    'max_gap_buckets': MAX_GAP_BUCKETS, 'post_scale_clip': POST_SCALE_CLIP,
    'scaler_flow': scaler_flow, 'scaler_bucket': scaler_bucket,
    'train_medians': train_medians,
    'feat_names_before_var': feat_names_before_var,
    'keep_var_mask': var_mask, 'drop_corr': drop_corr,
    'clip_bounds': clip_bounds, 'feat_cols_final': feat_cols_final,
    'bucket_feature_cols': bucket_feature_cols,
    'flow_count_max': flow_count_max, 'bucket_fill_values': bucket_fill,
}, pkl_path)
print(f'Saved preproc  -> {pkl_path}')

manifest = {
    'version'              : VERSION,
    'created_at'           : datetime.now(timezone.utc).isoformat(),
    'random_state'         : RANDOM_STATE,
    'scaler_type'          : 'StandardScaler+clip(flow & bucket)',
    'bucket_agg'           : BUCKET_AGG,
    'post_scale_clip'      : POST_SCALE_CLIP,
    'attack_frac_threshold': ATTACK_FRAC_THRESHOLD,
    'max_gap_buckets'      : MAX_GAP_BUCKETS,
    'shapes': {
        'X_train': list(X_train.shape),
        'X_val'  : list(X_val.shape),
        'X_test' : list(X_test.shape),
    },
    'attack_rates'         : {'val': float(y_val.mean()), 'test': float(y_test.mean())},
    'window_timestamps'    : True,
    'ts_test_unix_range'   : [int(ts_test.min()), int(ts_test.max())],
    'window_size'          : WINDOW_SIZE,
    'stride'               : STRIDE,
    'bucket_freq'          : BUCKET_FREQ,
    'features_per_bucket'  : len(bucket_feature_cols),
    'scale_stats'          : {'mean_abs': _xmean, 'max_abs': _xmax, 'nan': _nan},
    'leakage_free'         : True,
    'multiclass_labels'    : True,
    'multiclass_labels_val_unique' : sorted(np.unique(ymc_val).tolist()),
    'multiclass_labels_test_unique': sorted(np.unique(ymc_test).tolist()),
}
with open(f'{OUTPUT_DIR}/manifest_{VERSION}.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print(f'Saved manifest -> {OUTPUT_DIR}/manifest_{VERSION}.json')

# Official label map (Task 10; embedded, no separate .py)
_LABEL_MAP = {
    '0': 'BENIGN', '1': 'CVE-2020-13379', '2': 'Node-RED Recon',
    '3': 'Node-RED RCE', '4': 'Node-RED Escape', '5': 'CVE-2021-43798',
    '6': 'CVE-2019-20933', '7': 'CVE-2021-30465', '8': 'CVE-2021-25741',
    '9': 'CVE-2022-23648', '10': 'CVE-2019-5736', '11': 'DSB Nuclei Scan',
}
_map_path = f'{OUTPUT_DIR}/mdc_label_map.json'
with open(_map_path, 'w', encoding='utf-8') as fh:
    json.dump({
        'label_names': _LABEL_MAP,
        'citation': 'Sever & Dogan (2023), ITU Journal / MDC Kaggle dataset',
        'source': 'embedded in the merged preprocessing notebook',
    }, fh, indent=2)
print(f'Saved label map -> {_map_path}')

print()
print('This single notebook now produces everything downstream needs:')
print(f'  {Path(npz_path).name}, {Path(npz_mc_path).name}, {Path(drift_path).name},')
print(f'  preproc_{VERSION}.pkl, manifest_{VERSION}.json, mdc_label_map.json')


## 16. Zip outputs for download (Kaggle -- no Drive)


In [ ]:
# --- Kaggle IO helpers (self-contained -- no Drive, no Colab) ---
import os, sys, glob, shutil, zipfile, base64
from pathlib import Path

_IN_KAGGLE = os.path.exists('/kaggle/working')

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')

def kaggle_find(name, extra_dirs=()):
    """Search /kaggle/input/**, /kaggle/working/**, and extra_dirs for a file by
    name. There is no live shared Drive on Kaggle -- to hand a file from one
    notebook to the next, either (a) attach the producing notebook's own
    output via '+ Add Data > Your Notebooks' (no manual zip needed), or
    (b) download this notebook's output zip and upload it as a new Kaggle
    Dataset, then attach that dataset. Either way it shows up under
    /kaggle/input/<name>/ and this function finds it there."""
    roots = [KAGGLE_INPUT, KAGGLE_WORKING, *[Path(d) for d in extra_dirs]]
    for root in roots:
        if not root.is_dir():
            continue
        hits = sorted(glob.glob(str(root / '**' / name), recursive=True))
        if hits:
            return Path(hits[0])
    return None

def kaggle_upload_fallback(name, dest_dir):
    """Best-effort interactive upload widget for a live session. Not available
    during a headless 'Save & Run All' commit (no UI) -- attach the file as
    input data instead in that case."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
        uploader = widgets.FileUpload(accept='', multiple=False)
        display(uploader)
        print(f'Upload {name} with the widget above, then re-run this cell.')
        if uploader.value:
            item = list(uploader.value.values())[0]
            content = item['content'] if isinstance(item, dict) else item.content
            dest = Path(dest_dir) / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(bytes(content))
            print(f'Saved -> {dest}')
            return dest
    except Exception as e:
        print(f'Interactive upload unavailable ({e}).')
    return None

def zip_and_offer_download(src_dir, zip_name, max_auto_mb=25):
    """Zip src_dir into /kaggle/working/{zip_name}.zip and try to trigger a
    browser download automatically. Only fires in a live, actively-open
    browser tab (not during headless 'Save & Run All') -- Kaggle's own Output
    tab (right sidebar) always lists this zip for manual download regardless
    of whether the auto-download trick actually fires in your browser."""
    src_dir = Path(src_dir)
    zip_base = KAGGLE_WORKING / zip_name
    zip_path = zip_base.with_suffix('.zip')
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_base), 'zip', root_dir=src_dir)
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f'Zipped -> {zip_path}  ({size_mb:.1f} MB)')
    if size_mb <= max_auto_mb:
        try:
            from IPython.display import HTML, display
            b64 = base64.b64encode(zip_path.read_bytes()).decode()
            html = (
                f'<a id="dl_{zip_name}" download="{zip_path.name}" '
                f'href="data:application/zip;base64,{b64}"></a>'
                f'<script>document.getElementById("dl_{zip_name}").click();</script>'
            )
            display(HTML(html))
            print('Auto-download triggered (only works if this tab is actively open --')
            print('if nothing happened, use the Output tab on the right instead).')
        except Exception as e:
            print(f'Auto-download trick failed ({e}) -- use the Output tab instead.')
    else:
        print(f'{size_mb:.1f} MB exceeds the {max_auto_mb} MB auto-download guard -- '
              'skipping the browser trick to avoid bloating notebook output.')
        print('Get it from the Output tab (right sidebar) after Save Version instead.')
    return zip_path

print(f'Kaggle IO ready. In Kaggle: {_IN_KAGGLE}')

npz_path = Path(OUTPUT_DIR) / 'windows_vnext.npz'
if not npz_path.is_file():
    raise FileNotFoundError(f'Missing {npz_path} -- run the save cell first')

zip_and_offer_download(OUTPUT_DIR, 'windows_vnext_processed')
print()
print('Contents (all artifacts for every downstream notebook):')
for p in sorted(Path(OUTPUT_DIR).glob('*')):
    print(' ', p.name, p.stat().st_size, 'bytes')
